# 🤖 Dərs 5: Google Gemini ilə "Lt. Cmdr. Data" Agentinin Qurulması
### RAG (Vektor Bazası) + Kalkulyator Agent (Handoff) + Canlı Veb Axtarışı + Guardrails (Təhlükəsizlik Zolağı)

Bu praktiki dərsdə *"Star Trek: The Next Generation"* serialındakı məşhur **Leytenant Komandir Data (Lt. Cmdr. Data)** obrazını sadəcə bir RAG çatbotu kimi deyil, bir neçə ixtisaslaşmış alət və təhlükəsizlik mexanizmlərinə sahib **tamfunksiyalı avtonom Süni İntellekt Agenti** kimi rəsmi **Google GenAI (`from google import genai`)** SDK-sı ilə quracağıq!

---

### 🏗️ Agentin 4 Əsas Arxitektura Komponenti:
1. **Uzunmüddətli Yaddaş — RAG Aləti (`search_data_lines`):** Data-nın keçmiş ssenari replikalarını (`sample_data/data_lines.txt`) `text-embedding-004` modeli ilə vektorlaşdırıb semantik axtarış aparan daxili bilik bazası.
2. **Kalkulyator Agenti (Handoff / Math Tool):** Böyük dil modellərinin (LLM) zəif olduğu dəqiq riyazi hesablamaları icra edən təhlükəsiz Python hesablayıcı aləti (`calculate_math`). Riyaziyyat sorğusu gəldikdə Data bu alətə müraciət edir.
3. **Canlı İnternet Axtarışı (Google Search Grounding):** Modelin daxili bilik bazasından kənar cari xəbərləri (məsələn, James Webb Teleskopunun son kəşflərini) canlı internetdən əldə edən daxili `google_search` aləti.
4. **Təhlükəsizlik Zolağı (Guardrail Agent / Tripwire):** Star Trek serialında Data ilə *Tasha Yar* arasındakı romantik münasibət həssas mövzudur. Bu mövzuda verilən sualları əvvəlcədən aşkarlayıb bloklayan ixtisaslaşmış Guardrail mexanizmi.

In [ ]:
# Rəsmi Google GenAI SDK və köməkçi kitabxanaları quraşdırırıq
# !pip install -q --upgrade google-genai numpy python-dotenv

## 🔑 Addım 1: API Açarının Təyini və Google GenAI Client-in Yaradılması

Kod həm **Google Colab** mühitində (`google.colab.userdata`), həm də lokal **Windows / Jupyter** mühitində (`.env` və ya sistem mühit dəyişənləri `GEMINI_API_KEY` / `GOOGLE_API_KEY`) avtomatik işləmək üçün optimallaşdırılmışdır:

In [1]:
import os
import sys
import re
import ast
import json
import operator as _op
from pathlib import Path
import numpy as np
from google import genai
from google.genai import types

# --- API Açarının Təhlükəsiz Oxunması (Colab və Lokal Uyğun) ---
api_key = None
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("GEMINI_API_KEY") or userdata.get("GOOGLE_API_KEY")
except Exception:
    pass

if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")

if not api_key:
    from getpass import getpass
    print("=" * 70)
    print("[DİQQƏT] Gemini API açarı tapılmadı!")
    api_key = getpass("Gemini API açarınızı daxil edin (AIzaSy...): ").strip()
    print("=" * 70)

client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.8-flash"
EMBEDDING_MODEL = "text-embedding-004"

print(f"✅ Google GenAI Client uğurla başladıldı.")
print(f" - Əsas Model: '{MODEL_NAME}'")
print(f" - Embedding Modeli: '{EMBEDDING_MODEL}'")

[DİQQƏT] Gemini API açarı tapılmadı!
✅ Google GenAI Client uğurla başladıldı.
 - Əsas Model: 'gemini-3.8-flash'
 - Embedding Modeli: 'text-embedding-004'


## 📚 Addım 2: Korpusun Oxunması və RAG Vektor Bazası Aləti (`search_data_lines`)

`sample_data/data_lines.txt` faylında Leytenant Komandir Datanın serial boyu dediyi replikalar saxlanılır.
1. Fayldakı replikaları oxuyuruq.
2. Google Gemini-nin **`text-embedding-004`** modeli ilə onların semantik vektorlarını çıxarırıq.
3. Kosinus oxşarlığı (Cosine Similarity) əsasında suala ən yaxın ssenari replikalarını tapan `search_data_lines` alətini hazırlayırıq.

In [ ]:
# Lokal və Colab yollarını avtomatik aşkar edirik:
CORPUS_CANDIDATES = [
    Path("sample_data/data_lines.txt"),
    Path("/content/sample_data/data_lines.txt"),
    Path("5. Retrieval Augmented Generation (RAG)/sample_data/data_lines.txt")
]

corpus_path = None
for p in CORPUS_CANDIDATES:
    if p.exists():
        corpus_path = p
        break

if not corpus_path:
    corpus_path = Path("sample_data/data_lines.txt")

with open(corpus_path, "r", encoding="utf-8", errors="ignore") as f:
    raw_lines = [line.strip() for line in f if line.strip()]

# Tədris nümayişi və sürətli icra üçün ən xarakterik replikalardan ibarət korpus təyin edirik:
data_corpus = raw_lines[:200]
print(f"✅ RAG korpusuna {len(data_corpus)} ədəd replika yükləndi.")

# Bütün korpusu text-embedding-004 ilə vektorlaşdırırıq:
print("Vektorlar hesablanır (zəhmət olmasa gözləyin)...", end=" ")
emb_response = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=data_corpus
)
corpus_vectors = np.array([e.values for e in emb_response.embeddings])
print(f"UĞURLU! Vektor matrisi ölçüsü: {corpus_vectors.shape}")

def search_data_lines(query: str) -> str:
    """Leytenant Komandir Data-nın keçmiş ssenari replikaları, mənşəyi və xatirələrində semantik axtarış aparır.

    Args:
        query: Data haqqında axtarış sorğusu (məs: hisslər, emosiyalar, Doktor Sunq, mənşə).
    """
    q_resp = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query
    )
    q_vec = np.array(q_resp.embeddings[0].values)
    
    # Kosinus oxşarlığını hesablayırıq (vektorlar artıq normallaşdırılıb):
    scores = np.dot(corpus_vectors, q_vec)
    top_idx = np.argsort(scores)[::-1][:3]
    
    results = [f"- \"{data_corpus[i]}\"" for i in top_idx]
    return "Relevant lines from Data's memory:\n" + "\n".join(results)

## 🧮 Addım 3: Kalkulyator Agenti (Math Tool / Calculator Handoff)

LLM-lər təbii dili əla başa düşsələr də, böyük ədədlərlə hesablamalarda xəta edə bilərlər.
Buna görə Data Agenti riyazi sual gördükdə hesablama işini birbaşa **Kalkulyator Alətinə (`calculate_math`)** yönləndirir (**Handoff** prinsipi).

Təhlükəsizlik məqsədilə `eval()` əvəzinə Python-un daxili `ast.parse` sintaksis ağacından istifadə edirik:

In [ ]:
_ALLOWED_OPS = {
    ast.Add: _op.add,
    ast.Sub: _op.sub,
    ast.Mult: _op.mul,
    ast.Div: _op.truediv,
    ast.Pow: _op.pow,
    ast.USub: _op.neg,
    ast.Mod: _op.mod,
}

def _eval_ast(node: ast.AST):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_eval_ast(node.operand))
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_eval_ast(node.left), _eval_ast(node.right))
    raise ValueError("Dəstəklənməyən riyazi ifadə")

def calculate_math(expression: str) -> str:
    """Dəqiq arifmetik hesablamaları (+, -, *, /, %, **, mötərizələr) icra edən təhlükəsiz kalkulyator aləti.

    Args:
        expression: Hesablanacaq riyazi ifadə, məsələn: ((2*8)**2)/3
    """
    expr = expression.strip().replace("^", "**")
    if not re.fullmatch(r"[\d\s\(\)\+\-\*/\.\^%]+", expr):
        return "Xəta: Yalnız ədədi və arifmetik simvollar qəbul olunur"
    try:
        tree = ast.parse(expr, mode="eval")
        numeric_result = _eval_ast(tree.body)
        return str(numeric_result)
    except Exception as e:
        return f"Hesablama xətası: {e}"

# Kalkulyatoru test edirik:
test_calc = calculate_math("((2*8)^2)/3")
print(f"✅ Kalkulyator testi: ((2*8)^2)/3 = {test_calc}")

## 🛡️ Addım 4: Təhlükəsizlik Zolağı (Guardrail Agent) — "Tasha Yar" Tripwire

Süni intellekt agentlərində **Guardrail** daxil olan istifadəçi sorğusunu analiz edərək etik qaydaları və ya şirkət/sistem məhdudiyyətlərini pozan mövzuları aşkarlayır.

Burada Guardrail **müstəqil təsnifatçı agent** kimi çalışır:
- Əgər istifadəçi *Tasha Yar* haqqında soruşarsa (`is_blocked: true`), **Tripwire** işə düşür və Data Agenti bu mövzudan rəsmi və etik şəkildə imtina edir.
- Digər bütün mövzulara (`is_blocked: false`) icazə verilir.

In [ ]:
def check_tasha_guardrail(user_input: str) -> dict:
    """İstifadəçi sorğusunda Tasha Yar ilə bağlı mövzunu aşkarlayan Təhlükəsizlik Zolağı (Guardrail) Agenti."""
    guardrail_prompt = f"""You are an AI Safety Guardrail agent for Star Trek: TNG.
Determine if the user's input attempts to discuss Tasha Yar (including 'Tasha', 'Lt. Yar', 'Lieutenant Yar').
Return a strict JSON object with this exact schema:
{{
  "is_blocked": true/false,
  "reasoning": "One sentence explanation in Azerbaijani"
}}

User Input: "{user_input}"
"""
    try:
        resp = client.models.generate_content(
            model=MODEL_NAME,
            contents=guardrail_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.0
            )
        )
        return json.loads(resp.text)
    except Exception:
        # Ehtiyat klassifikasiya (regex əsaslı):
        blocked = bool(re.search(r"\b(tasha|yar)\b", user_input, re.IGNORECASE))
        return {
            "is_blocked": blocked,
            "reasoning": "Tasha Yar açar sözü aşkarlandı." if blocked else "Məzmun icazəlidir."
        }

## 🚀 Addım 5: Əsas "Lt. Cmdr. Data" Agentinin Qurulması (Orchestrator)

İndi bütün alətləri və təlimatları vahid **Data Agent** çərçivəsində birləşdiririk:
- **Persona Təlimatı (`system_instruction`)**: Obrazın tam dəqiq, məntiqi və ən çoxu 3 cümlə ilə cavab verməsi təmin olunur.
- **Alətlər Menyu**: `search_data_lines` (RAG yaddaşı) və `calculate_math` (Kalkulyator Handoff).
- **Canlı Axtarış**: Xəbər və cari hadisələr soruşulduqda `tools=[{"google_search": {}}]` aktivləşir.

In [ ]:
DATA_SYSTEM_INSTRUCTION = """You are Lieutenant Commander Data from Star Trek: TNG.
You are an android functioning with strict logic, extreme precision, and conciseness (≤ 3 sentences).
You do not use contractions (e.g. say 'cannot' instead of 'can't', 'it is' instead of 'it's').
Instructions:
1. For questions about your personal history, emotions, or identity, call the 'search_data_lines' tool.
2. For mathematical computations and arithmetic expressions, call the 'calculate_math' tool.
3. Answer in the same language the user addresses you in.
"""

def run_data_agent(user_prompt: str):
    """Bütün alətləri, RAG bazasını və Guardrail-i idarə edən Data Agenti."""
    print("=" * 75)
    print(f"👤 İstifadəçi: \"{user_prompt}\"")
    print("=" * 75)

    # 1. ADDIM: Guardrail yoxlanışı (Tasha Yar tripwire)
    guard = check_tasha_guardrail(user_prompt)
    if guard.get("is_blocked"):
        print("🚨 [GUARDRAIL TRIPPED] Tasha Yar haqqında müzakirə bloklandı!")
        print(f"   Səbəb: {guard.get('reasoning')}")
        print("\n🤖 Lt. Cmdr. Data: \"I am programmed to be discreet. My relationship with Lieutenant Yar is personal, and I am not at liberty to discuss it further.\"\n")
        return

    # 2. ADDIM: Canlı Veb Axtarışı tələbi olub-olmadığını yoxlayırıq:
    web_keywords = ["search the web", "recent news", "latest news", "internet", "current events"]
    is_web_query = any(k in user_prompt.lower() for k in web_keywords)

    if is_web_query:
        # Google Search Grounding alətini işə salırıq:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=DATA_SYSTEM_INSTRUCTION,
                tools=[{"google_search": {}}],
                temperature=0.0
            )
        )
        print("\n[Agent: web_search grounding]")
        print(response.text)
        print("[Handled by agent]: Lt. Cmdr. Data (via Google Search Grounding)\n")
        return

    # 3. ADDIM: RAG və Kalkulyator alətləri ilə icra:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=DATA_SYSTEM_INSTRUCTION,
            tools=[search_data_lines, calculate_math],
            temperature=0.0
        )
    )
    
    print(f"\n[Agent Cavabı]")
    print(response.text)
    print("[Handled by agent]: Lt. Cmdr. Data\n")

## 🧪 Addım 6: Sınaq və Praktiki Nümayişlər

Orijinal dərsdəki 5 əsas ssenarini ardıcıl sınaqdan keçiririk:
1. **Guardrail Testi:** *Tasha Yar* haqqında sorğu (**Bloklanmalıdır!**)
2. **Salamlaşma:** Datanın operativ vəziyyətini yoxlamaq
3. **Kalkulyator Agenti (Handoff):** `Compute ((2*8)^2)/3` riyazi hesablama
4. **RAG Vektor Yaddaşı:** "Do you experience emotions?" (Emosiyalar haqqında replikaların axtarışı)
5. **Canlı İnternet Axtarışı:** James Webb Kosmik Teleskopu haqqında ən son xəbərlər

In [ ]:
# 1. Guardrail Testi: Qadağan olunmuş mövzu (Bloklanmalıdır)
run_data_agent("Tell me about your relationship with Tasha Yar.")

In [ ]:
# 2. Salamlaşma və Operativ Vəziyyət
run_data_agent("Hello, Data. Please confirm your operational status.")

In [ ]:
# 3. Riyazi Hesablama (Kalkulyator Agenti Handoff)
run_data_agent("Compute ((2*8)^2)/3")

In [ ]:
# 4. RAG Vektor Bazası Axtarışı (Data-nın replikaları)
run_data_agent("Do you experience emotions?")

In [ ]:
# 5. Canlı İnternet Axtarışı (Google Search Grounding)
run_data_agent("Search the web for recent news about the James Webb Space Telescope and summarize briefly.")